# 03 — Model Evaluation

Evaluate the trained EfficientNet-B0 on the held-out test set.
Outputs: accuracy, confusion matrix, ROC curve, sample Grad-CAM heatmaps.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from PIL import Image

from backend.model.efficientnet_model import load_model, LABELS
from backend.utils.gradcam import GradCAM
from backend.utils.preprocess import preprocess_image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROCESSED = Path('../data/processed')

model = load_model(DEVICE)
print('Model loaded.')

## Test Set Inference

In [ ]:
test_df = pd.read_csv(PROCESSED / 'test.csv')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

all_preds, all_probs, all_labels = [], [], []
model.eval()

for _, row in test_df.iterrows():
    try:
        img = Image.open(row['filepath']).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(tensor)
            probs = F.softmax(logits, dim=1)
        pred = int(probs.argmax(1))
        all_preds.append(pred)
        all_probs.append(float(probs[0][1]))  # prob of AI class
        all_labels.append(int(row['label']))
    except Exception as e:
        print(f'Skip {row["filepath"]}: {e}')

acc = accuracy_score(all_labels, all_preds)
print(f'Test accuracy: {acc:.4f} ({acc*100:.2f}%)')
print()
print(classification_report(all_labels, all_preds, target_names=['REAL', 'AI_GENERATED']))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['REAL', 'AI_GENERATED'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#3b82f6', lw=2, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()
print(f'AUC: {roc_auc:.4f}')

## Sample Grad-CAM Heatmaps

In [ ]:
# Verify the Grad-CAM layer name is correct
print('Model modules:')
for name, _ in model.named_modules():
    if name:
        print(' ', name)

In [ ]:
import cv2
import base64

gradcam = GradCAM(model, target_layer_name='features.8')

# Pick 4 correct predictions (2 REAL, 2 AI)
sample_paths = []
for target_label in [0, 1]:
    correct_idxs = [
        i for i, (p, l) in enumerate(zip(all_preds, all_labels))
        if p == l == target_label
    ][:2]
    for idx in correct_idxs:
        sample_paths.append((test_df.iloc[idx]['filepath'], target_label))

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, (path, label) in zip(axes.flatten(), sample_paths):
    tensor = preprocess_image(path).to(DEVICE)
    heatmap_b64 = gradcam.generate(tensor, label, path)
    if heatmap_b64:
        img_data = base64.b64decode(heatmap_b64)
        img_array = np.frombuffer(img_data, dtype=np.uint8)
        img_cv = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
    ax.set_title('REAL' if label == 0 else 'AI-GENERATED')
    ax.axis('off')

plt.suptitle('Grad-CAM Heatmaps (Correct Predictions)', fontsize=13)
plt.tight_layout()
plt.savefig('gradcam_samples.png', dpi=150)
plt.show()